# Bayesian Spatial Synthetic Control in Python## California's Proposition 99 with `scspill` and `mlsynth`**Carlos Mendez** — Nagoya University (GSID)Companion notebook for [the full tutorial](https://carlos-mendez.org/post/python_sc_bayes_spatial/).---Three nested estimators on one panel, each relaxing one assumption of the last:1. **Classical synthetic control** — donor weights on the simplex (Abadie, Diamond & Hainmueller 2010)2. **Bayesian synthetic control** — the simplex replaced by a horseshoe prior3. **Bayesian spatial synthetic control** — SUTVA on the donor pool dropped (Sakaguchi & Tagawa 2026)The question the third stage can ask and the first two cannot: **who else was treated?**> **Budget note.** This notebook runs at `m_iter=4000` so a first pass finishes in a couple of> minutes. The post's headline numbers use **500,000** iterations. The tutorial budget reproduces> the *shape* of every result — signs, ranks, orders of magnitude — but not the third decimal, and> its effective sample size for $\rho$ is far below anything publishable. Section 9 shows why that> matters more than it sounds.

## 0. SetupAbout two minutes the first time: one PyPI wheel and one git clone.

In [ ]:
# Both pins matter. scspill 0.2.1 is the release every number in the post was# produced under. mlsynth is pinned to a COMMIT because its PyPI release lags# main at the same version string.%pip install -q "scspill[numba]==0.2.1"%pip install -q "mlsynth[bayes] @ git+https://github.com/jgreathouse9/mlsynth.git@15f168bb90487098a7324be00b6663fcab0139ef"

In [ ]:
# Colab sometimes downgrades numpy while resolving. Check before continuing.import importlib.metadata as mdfor p in ("scspill", "mlsynth", "numpy", "pandas", "scipy", "matplotlib", "numba", "numpyro"):    try:        print(f"{p:12s} {md.version(p)}")    except Exception:        print(f"{p:12s} not installed")import numpy as npif np.__version__.startswith("1."):    print("\n*** RESTART REQUIRED ***")    print("numpy was downgraded during install. Runtime > Restart session, then re-run"          " from this cell (do NOT re-run the install cell).")

In [ ]:
import os# BLAS reduction order changes the last digits; at N = 38 single-threaded is# also faster. Must precede the numpy import to have any effect.for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):    os.environ.setdefault(v, "1")os.environ.setdefault("JAX_ENABLE_X64", "1")import warningswarnings.filterwarnings("ignore")import matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport mlsynthimport scspillfrom scspill import SCSPILLfrom scspill.data import load_californiaSEED = 20251022          # the R edition's seed, so the two are comparableTREAT_YEAR = 1988M_ITER, BURN = 4_000, 2_000# The R edition's published numbers, for comparison throughout.R_EDITION = {"classical": -18.46, "horseshoe": -15.84, "sar": -16.59,             "rho": 0.2226, "rho_ess": 2.93, "nevada": -3.75}plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (9, 5),                     "axes.grid": True, "grid.alpha": 0.3})print(f"scspill {scspill.__version__}   mlsynth {mlsynth.__version__}")

## 1. The data`scspill` ships the Abadie–Diamond–Hainmueller Proposition 99 panel **and** the two spatial objectsthe third stage needs, so there is nothing to download and nothing to merge.

In [ ]:
panel = load_california()df = panel.df.copy()donors = list(panel.spatial_W.index)print(panel.description)print()print(df.head())print(f"\nshape {df.shape}   states {df['state'].nunique()}   "      f"years {df['year'].min()}-{df['year'].max()}   missing {df.isna().sum().sum()}")

### The two spatial objects- `spatial_w` — how exposed each **donor** is to the **treated** unit.- `spatial_W` — donor-to-donor contiguity, row-normalised inside the estimator.`spatial_w` has exactly one non-zero entry.

In [ ]:
print("California's neighbours inside the donor pool:")print(panel.spatial_w[panel.spatial_w > 0])W = panel.spatial_W.loc[donors, donors]print(f"\nW {W.shape}   symmetric {np.allclose(W.values, W.values.T)}")print(f"degree: min {W.sum(1).min():.0f}  max {W.sum(1).max():.0f}  mean {W.sum(1).mean():.2f}")

Oregon and Arizona also border California, but neither is in the donor pool — both were excluded forrunning their own tobacco-control programmes. So there is **exactly one leak channel**, and we canname it.### Treatment in 1988, not 1989Proposition 99 passed in November 1988; the tax took effect on 1 January 1989. ADH and `mlsynth`'sown example use 1989. `scspill` uses **1988**, following the R replication package. Neither iswrong, but they cannot be mixed — pin one so every stage sees the same $T_0$.

In [ ]:
rebuilt = ((df["state"] == "California") & (df["year"] >= TREAT_YEAR)).astype(int)assert (rebuilt.to_numpy() == df["treated"].to_numpy()).all()years = np.sort(df["year"].unique())wide = df.pivot(index="year", columns="state", values="cigsale")y_treated = wide["California"].to_numpy()post = years >= TREAT_YEARprint(f"T0 = {(~post).sum()}   T1 = {post.sum()}   donors = {len(donors)}")

In [ ]:
fig, ax = plt.subplots()for s in donors:    ax.plot(years, wide[s], color="#8b9dc3", lw=0.7, alpha=0.5)ax.plot(years, y_treated, color="#d97757", lw=2.6, label="California")ax.axvline(TREAT_YEAR - 0.5, color="grey", ls="--")ax.plot([], [], color="#8b9dc3", label=f"{len(donors)} donor states")ax.set(xlabel="Year", ylabel="Cigarette sales (packs per capita)",       title="California leaves the pack after 1988")ax.legend()plt.show()

California is *already* declining faster than the pack well before the dashed line. That is whydifference-in-differences will not do here — parallel trends is visibly false — and why syntheticcontrol exists.## 2. Stage 1 — classical simplex synthetic controlChoose weights $\alpha$ to make a blend of donors track California before 1988:$$\widehat{\alpha} = \arg\min_{\alpha \in \Delta} \sum_{t \le T_0} \Big(Y_{1t} - \alpha^{\top}\mathbf{Y}^{c}_{t}\Big)^2,\qquad \Delta = \Big\{\alpha : \alpha_j \ge 0, \; \sum_j \alpha_j = 1\Big\}$$The set $\Delta$ is the **simplex**, and everything that separates the three stages is a statementabout it.

In [ ]:
common = dict(df=df, outcome="cigsale", treat="treated",              unitid="state", time="year", display_graphs=False)sc = mlsynth.VanillaSC(dict(common)).fit()w_sc = pd.Series(sc.donor_weights).reindex(donors).fillna(0.0)print(f"ATT                : {sc.att:.4f}   (R edition {R_EDITION['classical']:.2f})")print(f"pre-treatment RMSE : {sc.pre_rmse:.4f}")print(f"weights sum        : {w_sc.sum():.6f}")print(f"active donors      : {(w_sc > 1e-4).sum()} of {len(donors)}\n")print(w_sc[w_sc > 1e-4].sort_values(ascending=False).round(4).to_string())

Five of 38 donors carry the whole counterfactual and 33 get exactly zero. Two things to notice:1. This reproduces the R edition to **0.04 packs**, using a different package and a different   optimiser. Two independent implementations landing this close is the strongest evidence either   gets that the estimator is correctly coded.2. **Nevada is in the blend, with a weight of about 0.24** — the one state we have a specific reason   to suspect of contamination is carrying a quarter of the counterfactual.

In [ ]:
cf_sc = np.asarray(sc.time_series.counterfactual_outcome, float).ravel()fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(9, 7),                             gridspec_kw={"height_ratios": [2, 1]})a1.plot(years, y_treated, color="#6a9bcc", lw=2.2, label="California")a1.plot(years, cf_sc, color="#00d4c8", lw=2, ls="--", label="Synthetic California")a1.axvline(TREAT_YEAR - 0.5, color="grey", ls="--"); a1.legend()a1.set(ylabel="Packs per capita", title="Stage 1: classical synthetic control")a2.plot(years, y_treated - cf_sc, color="#d97757", lw=2)a2.axhline(0, color="grey"); a2.axvline(TREAT_YEAR - 0.5, color="grey", ls="--")a2.set(xlabel="Year", ylabel="Gap")plt.show()

## 3. Stage 2 — Bayesian synthetic controlReplace the hard constraint with a **prior that prefers zero without forbidding anything else**.The horseshoe prior (Carvalho, Polson & Scott 2010) has an infinite spike at zero and Cauchy tails:$$\alpha_j \mid \lambda_j \sim \mathcal{N}\big(0, \lambda_j^2\big), \qquad\lambda_j \mid \tau \sim \mathcal{C}^{+}(0, \tau), \qquad\tau \sim \mathcal{C}^{+}(0, \sigma)$$Most $\lambda_j$ come out tiny — shrinking that donor to nothing — while any single one can beenormous if the likelihood insists.

In [ ]:
bscm = mlsynth.BSCM({**common, "prior": "horseshoe", "n_iter": 8_000,                     "burn_in": 4_000, "chains": 4, "seed": SEED}).fit()w_bscm = pd.Series(bscm.donor_weights).reindex(donors).fillna(0.0)beta0 = float(np.mean(np.asarray(bscm.posterior.beta0)))print(f"ATT           : {bscm.att:.4f}")print(f"95% CrI       : [{bscm.att_ci[0]:.4f}, {bscm.att_ci[1]:.4f}]")print(f"intercept     : {beta0:.4f}   <- BSCM fits one; scspill does not")print(f"weights sum   : {w_bscm.sum():.4f}   <- not 1, because of that intercept")print(f"active donors : {(w_bscm.abs() > 0.01).sum()} of {len(donors)}")

Hold on to the intercept. In a moment `scspill` will report a Bayesian synthetic control about**3 packs away** from this one, and the intercept is the entire explanation:$$\text{BSCM:} \quad Y_{1t} = \beta_0 + \sum_j \alpha_j Y_{jt} + \varepsilon_t\qquad\text{versus}\qquad\text{scspill:} \quad Y_{1t} = \sum_j \alpha_j Y_{jt} + \varepsilon_t$$Neither is more correct in the abstract. They answer different questions, and averaging them wouldbe meaningless.## 4. Stage 3 — Bayesian spatial synthetic controlNow drop SUTVA. Each donor's outcome is allowed to depend on its neighbours' outcomes *and* onCalifornia's, with one intensity parameter $\rho$:$$\mathbf{Y}^{c}_{t} = \rho\big(\mathbf{w}\,Y_{1t} + W\mathbf{Y}^{c}_{t}\big) + X_t\beta + \mathbf{u}_t$$With $A = W + \mathbf{w}\alpha^{\top}$, the donors' no-treatment outcomes solve in closed form,$$\mathbf{Y}^{c}_{t}(\mathbf{0}) = \big(I - \rho A\big)^{-1}\Big[\big(I - \rho W\big)\mathbf{Y}^{c}_{t} - \rho\,\mathbf{w}\,Y_{1t}\Big]$$and **both** estimands follow — the effect on California and the spillover onto each donor:$$\xi_{0t} = Y_{1t} - \alpha^{\top}\mathbf{Y}^{c}_{t}(\mathbf{0}), \qquad\boldsymbol{\xi}^{c}_{t} = \mathbf{Y}^{c}_{t} - \mathbf{Y}^{c}_{t}(\mathbf{0})$$Note what is **absent** from those two lines: no $\beta$, no factors, no error variances. Only$(\alpha, \rho, \mathbf{w}, W)$ and the observed data. That cancellation is why a weakly identifiednuisance block does not poison the effects.Setting $\rho = 0$ returns Stage 2 exactly.

In [ ]:
result = SCSPILL({    **panel.config_kwargs(),      # df, columns, spatial_w, spatial_W, covariates    "m_iter": M_ITER,    "burn": BURN,    "seed": SEED,    "display_graphs": False,}).fit()print(f"ATT         : {result.att:.4f}  95% CrI "      f"[{result.att_ci[0]:.4f}, {result.att_ci[1]:.4f}]")print(f"ATT at rho=0: {result.effects_detail.att_scm:.4f}   <- Stage 2, free")print(f"rho         : {result.rho_hat:.4f}  95% CrI "      f"[{result.rho_ci[0]:.4f}, {result.rho_ci[1]:.4f}]")print(f"ESS(rho)    : {result.rho_ess:.1f}   acceptance {result.acc_rho:.3f}")print(f"\nR edition   : ATT {R_EDITION['sar']:.2f}  rho {R_EDITION['rho']:.4f}  "      f"ESS {R_EDITION['rho_ess']:.2f}")

**The credible interval for $\rho$ excludes zero.** That is the formal statement that the datareject the restriction collapsing Stage 3 back to Stage 2: SUTVA on the donor pool is not merelydoubtful here, it is rejected by the model that nests it.`scspill` ships its own diagnostics table, and it shows exactly where the weak identification lives.Read the `ess` column top to bottom.

In [ ]:
print(result.diagnostics(top_n_alpha=6).round(4))

$\sigma^2$ and the donor weights have effective sample sizes in the thousands. $\rho$ has a fractionof that, **from the same chain**. Everything in this model is easy except the one scalar the thirdstage exists to estimate — because there is exactly one contiguity channel out of California, so thepanel contains one state's worth of evidence about how strongly policies leak.### The package's own plots

In [ ]:
result.plot(kind="panel")plt.show()

In [ ]:
import matplotlib.pyplot as pltfig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))result.plot(kind="rho", ax=a)result.plot(kind="trace", ax=b)plt.tight_layout(); plt.show()

## 5. Who else was treated?The second estimand is a whole panel — one spillover per donor per year.Sign convention: `spillover_panel = Yc - Yc(0)`. **Negative** means the donor sold *fewer* packs thanit would have without Proposition 99.

In [ ]:
spill = result.spillover_panel.loc[TREAT_YEAR:]    # post-treatment rows onlyranked = spill.mean().reindex(spill.mean().abs().sort_values(ascending=False).index)print(ranked.head(6).round(4).to_string())print(f"\nNevada absorbs {abs(ranked.iloc[0]) / abs(ranked.iloc[1]):.1f}x the next-largest donor.")print(f"R edition: Nevada {R_EDITION['nevada']:.2f}")

The prior expectation was cross-border shopping *raising* Nevada's sales. The estimate says they**fell**. Whatever mechanism dominates — advertising, media, social norms crossing a border that taxarbitrage also crosses — the net effect on Nevada ran the same way as the effect on California.That direction has a consequence for the headline number, and it is the opposite of what most peopleguess.## 6. What the leak costA classical synthetic control built from *observed* donors reports$$Y_{1t} - \sum_j \alpha_j Y_{jt} = \underbrace{\xi_{0t}}_{\text{what we want}}- \underbrace{\sum_j \alpha_j \xi^{c}_{jt}}_{\text{bias}}$$so the bias is $-\sum_j \alpha_j \xi^{c}_{j}$ — a **product** of weight and spillover. A heavilycontaminated donor with zero weight is harmless; a lightly contaminated one carrying half thecounterfactual is not. Check the identity directly:

In [ ]:
alpha = pd.Series(result.alpha_hat, index=donors)xi = spill.mean()contrib = (alpha * xi).sort_values()print(contrib.head(4).round(4).to_string())print(f"\nsum_j alpha_j * xi_j             : {contrib.sum():+.4f}")print(f"att (purged) - att_scm (contam.) : "      f"{result.att - result.effects_detail.att_scm:+.4f}")

Nevada is almost the whole story. The spillovers are negative and the weights positive, so the biasis **positive**: the contaminated estimate is *closer to zero* than the truth. The classical andhorseshoe estimates **understate** Proposition 99's effect on California.And it is small — around a pack per capita out of seventeen. That is the honest summary: thespillover was real, statistically clear, and substantively modest **for California**. It was notmodest for Nevada, which is a different question, and the one classical synthetic control could nothave asked.## 7. The ladder

In [ ]:
ladder = pd.DataFrame([    ("1.  Classical SC (simplex)",        float(sc.att),   R_EDITION["classical"]),    ("2a. Bayesian SC (BSCM, intercept)", float(bscm.att), np.nan),    ("2b. Bayesian SC (scspill, rho=0)",  float(result.effects_detail.att_scm),     R_EDITION["horseshoe"]),    ("3.  Bayesian spatial SC",           float(result.att), R_EDITION["sar"]),], columns=["stage", "att", "r_edition"])ladder["diff"] = ladder["att"] - ladder["r_edition"]print(ladder.round(3).to_string(index=False))print(f"\nspread: {ladder['att'].max() - ladder['att'].min():.2f} packs; "      f"every stage agrees on the sign")

## 8. The rest of the catalogue`mlsynth` ships many more Bayesian and spillover estimators. The column that keeps a table like thisfrom lying is **`comparable`** — several of these target a *different* estimand, and reading themagainst the ladder above would be the easiest way to draw a false conclusion from a tidy-lookingtable.

In [ ]:
units = ["California"] + donorsW39 = pd.DataFrame(0.0, index=units, columns=units)W39.loc[donors, donors] = W.valuesW39.loc["California", donors] = panel.spatial_w.reindex(donors).to_numpy()W39.loc[donors, "California"] = panel.spatial_w.reindex(donors).to_numpy()SPECS = [    ("MVBBSC", mlsynth.MVBBSC,     dict(n_warmup=500, n_samples=500, n_chains=2, seed=SEED), True,     "ATT on the treated"),    ("SPILLSYNTH(sar)", mlsynth.SPILLSYNTH,     dict(method="sar", spatial_W=W, spatial_w=panel.spatial_w.reindex(donors),          p_factors=1, mcmc_iter=M_ITER, mcmc_burn=BURN, step_rho=0.01,          mcmc_seed=SEED), True,     "spillover-adjusted ATT -- the SAME paper, independently ported"),    ("SPILLSYNTH(cd)", mlsynth.SPILLSYNTH,     dict(method="cd", affected_units=["Nevada"]), False,     "measured against a DEMEANED leave-one-out baseline"),    ("SpSyDiD", mlsynth.SpSyDiD, dict(spatial_matrix=W39), False,     "reports direct, total and average indirect effects at once"),    ("ISCM", mlsynth.ISCM,     dict(inference=True, n_draws=1000, random_state=SEED), False,     "imperfect-fit correction on its own normalisation"),    ("SPOTSYNTH", mlsynth.SPOTSYNTH,     dict(selection="S1", forecast="loo", n_samples=1000, n_warmup=500,          seed=SEED), True,     "ATT after screening contaminated donors out of the pool"),]rows = []for name, cls, kw, comparable, estimand in SPECS:    try:        r = cls({**common, **kw}).fit()        rows.append(dict(estimator=name, att=round(float(r.att), 3),                         comparable="yes" if comparable else "NO",                         estimand=estimand))    except Exception as exc:            # a failure is information, not an error        rows.append(dict(estimator=name, att=np.nan, comparable="error",                         estimand=f"{type(exc).__name__}: {str(exc).splitlines()[0][:60]}"))print(pd.DataFrame(rows).to_string(index=False))

`SPILLSYNTH(method="sar")` is an **independent port of the same paper by a different author**,sharing no code with `scspill`. Its agreement with Stage 3 is the strongest external check eitherlibrary gets.## 9. Why the interval, not the point, is the hard partThe [R edition of this post](https://carlos-mendez.org/post/r_sc_bayes_spatial/) reports the sameATT to within a few tenths of a pack — and a 95% credible interval **0.38 packs wide**, against the12.7 packs the corrected configuration reports at the headline budget.`scspill` documents six departures from the authors' R replication code. Three have escape hatches,so we can put the Python code back into the R specification and watch what happens.

In [ ]:
rspec = SCSPILL({**panel.config_kwargs(), "m_iter": 5_000, "burn": 2_500,                 "seed": SEED, "display_graphs": False,                 "beta_prior": "ridge",       # departure 2                 "propagate_alpha": False,    # departure 3                 "adapt_rho": False,          # departure 4                 "step_rho": 0.01}).fit()rw = rspec.att_ci[1] - rspec.att_ci[0]cw = result.att_ci[1] - result.att_ci[0]print("R specification, at the R edition's own 5,000-iteration budget:")print(f"   ATT      {rspec.att:+.4f}   (R edition {R_EDITION['sar']:+.2f})")print(f"   rho      {rspec.rho_hat:.4f}    (R edition {R_EDITION['rho']:.4f})")print(f"   ESS(rho) {rspec.rho_ess:.2f}      (R edition {R_EDITION['rho_ess']:.2f})")print(f"\ninterval width: R spec {rw:.3f}  vs corrected {cw:.3f}  ({cw / rw:.0f}x wider)")

Independent code in a different language reproducing $\hat\rho$ to three decimals **including thepathology** — an effective sample size of 3 is the R sampler's behaviour, faithfully reproduced.Two distinct things were wrong with that interval, and they are worth keeping apart:- **Effective sample size asks whether the interval is *reliable*** — whether the chain visited  enough of the posterior for its quantiles to mean anything. At ESS 3, no.- **`propagate_alpha` asks whether the interval is *complete*** — whether it accounts for everything  the model is uncertain about. The R code varies $\rho$ while holding the donor weights fixed at  their posterior mean, so the reported interval contains **no** uncertainty about which states make  up synthetic California.Running the R specification for a hundred times as many iterations does *not* widen the interval.Propagating $\alpha$ does.**Report the effective sample size beside every credible interval, or the interval is decoration.**## 10. Where to go next- **[The full tutorial](https://carlos-mendez.org/post/python_sc_bayes_spatial/)** — every equation  derived, the six departures in detail, prior predictive and Geweke diagnostics, a Monte Carlo  study, and the evidence behind the 500,000-iteration budget.- **[The R edition](https://carlos-mendez.org/post/r_sc_bayes_spatial/)** — the same three stages  using the authors' own R and C++ code.- **[The synthetic control ladder in Python](https://carlos-mendez.org/post/python_sc_dsc_sdid/)** —  a different set of stages (DiD through synthetic DiD) on the Brexit referendum.- `scspill.data.load_sudan()` ships the paper's other application: 34 African countries, South  Sudan's 2011 secession, with exposure built from **bilateral trade** rather than borders. Dense  where contiguity was sparse, so $\rho$ should be far better identified. Check whether it is.### Exercises1. Rebuild `df["treated"]` at 1989 instead of 1988 and rerun all three stages. Is the change larger   or smaller than the gap between the simplex and the horseshoe?2. Zero out Nevada's entry in `panel.spatial_w` and refit. What happens to $\hat\rho$, to the ATT,   and to Idaho's and Utah's spillovers? This is the cleanest way to see how much of the story rests   on one entry in one vector.3. Raise `m_iter` to 100,000 and watch `ESS(rho)` and the interval width. How many draws would an   ESS of 400 take?